In [5]:
# Data 불러오기

import pandas as pd

train = pd.read_csv("../Data/train_20k.csv", header=None)
test = pd.read_csv("../Data/test_1k.csv", header=None)

In [6]:
# 예전에 결측치는 확인 했었어서 패스

In [8]:
# train의 data와 target 구분하기
train_data = train.iloc[:,1:]
train_target = train.iloc[:,:1]

In [9]:
# test의 data와 target 구분하기
test_data = test.iloc[:,1:]
test_target = test.iloc[:,:1]

#### Data들을 Tensor로 변환

In [10]:
import torch
train_input = torch.tensor(train_data.values) # values쓰면 numpy array 가 들어감
train_target = torch.tensor(train_target.values) # values쓰면 numpy array 가 들어감

test_input = torch.tensor(test_data.values)
test_target = torch.tensor(test_target.values)

print(train_input.data.shape)
print(train_target.data.shape)
print(test_input.data.shape)
print(test_target.data.shape)


# 넘파이 어레이를 (벨류)를 텐서로 바꾼 거

torch.Size([20001, 784])
torch.Size([20001, 1])
torch.Size([1001, 784])
torch.Size([1001, 1])


----
#### Data들의 차원 변경

In [11]:
train_input = train_input.reshape(-1, 28, 28)
train_target = train_target.reshape(-1,)
test_input = test_input.reshape(-1, 28, 28)
test_target = test_target.reshape(-1,)

print(train_input.data.shape)
print(train_target.data.shape)
print(test_input.data.shape)
print(test_target.data.shape)

torch.Size([20001, 28, 28])
torch.Size([20001])
torch.Size([1001, 28, 28])
torch.Size([1001])


In [12]:
# 데이터 표준화 및 2차원 행렬
train_scaled = (train_input / 255.0).reshape(-1, 28*28)
test_scaled = (test_input / 255.0).reshape(-1, 28*28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([20001, 784])
torch.Size([1001, 784])


#### 모델 만들기

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [14]:
# Train과 Valid
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(
                                                        train_scaled,
                                                        train_target,
                                                        test_size=0.2,
                                                        random_state=42
)

In [15]:
# Dataset과 Dataloader 생성

batch_size = 32 # mini batch
train_dataset = TensorDataset(train_scaled, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 한번 epoch가 발생했을때 섞어 쓰는 거

val_dataset = TensorDataset(val_scaled, val_target)
val_loader = DataLoader(val_dataset, batch_size=batch_size) # 벨리드나 테스트는 검증이라 셔플을 쓰지 않는다 

#### 모델 정의
: 입력층 -> 은닉층(활성화함수) -> 출력층으로 구분

In [16]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__() # super에 있는 모델을 쓰겠다는 거 
        self.flatten = nn.Flatten() # 층을 펴주는 거 
        self.fc1 = nn.Linear(28*28, 512)
        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(512,10) # 512개 들어와서 10개로 준다 
        self.softmax = nn.Softmax(dim=1) # 앞에는 변수임
    
    def forward(self, x) : 
        x = self.flatten(x) # 들어온 데이터로 층 만든것
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return self.softmax(x)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [21]:
# 모델 인스턴스
model = NeuralNetwork().to(device)

In [22]:
# 손실함수와 옵티마이져
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters()) # 모델에 파라메터를 가지고 쓰겠다 -> 모델의 가중치값을 가지고 변환 시키겠다 

#### 모델 훈련

In [23]:
# 학습 함수
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device) # 디바이스 (cpu)로 보냄
        optimizer.zero_grad() # 초기화 시켜주는 거 _옵티마이저가 곱하기 하는거라서 
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    return loss.item()

In [24]:
# 평가함수 
def evaluate(model,val_loader,criterion,device):
    model.eval()
    total_loss = 0 # 전체 손실 합계 
    correct = 0  # 정확하게 예측한 샘플 수 
    total = 0 # 전체 샘플 수
    with torch.no_grad():
        for inputs ,targets in val_loader: # 문제 정답 넣기
            inputs,targets = inputs.to(device),targets.to(device) # 문제 정답 디바이스로 보내기
            outputs = model(inputs)
            loss = criterion(outputs,targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss / len(val_loader), correct / total

In [25]:
# 예측 함수
def predict(model, data_loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            predictions.extend(predicted.cpu().numpy())
        return predictions

---
#### 학습 및 평가

In [26]:
model.to(device)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [27]:
# 훈련하기
num_epochs = 100

for epoch in range(num_epochs):
    train_loss = train(model,train_loader,criterion,optimizer,device)
    print(f'Epoch[{epoch+1:>3}/ {num_epochs}],Loss : {train_loss:.4f}')

Epoch[  1/ 100],Loss : 2.1859
Epoch[  2/ 100],Loss : 2.1823
Epoch[  3/ 100],Loss : 2.1834
Epoch[  4/ 100],Loss : 2.1902
Epoch[  5/ 100],Loss : 2.1774
Epoch[  6/ 100],Loss : 2.1774
Epoch[  7/ 100],Loss : 2.1716
Epoch[  8/ 100],Loss : 2.1843
Epoch[  9/ 100],Loss : 2.1772
Epoch[ 10/ 100],Loss : 2.1808
Epoch[ 11/ 100],Loss : 2.1733
Epoch[ 12/ 100],Loss : 2.1766
Epoch[ 13/ 100],Loss : 2.1716
Epoch[ 14/ 100],Loss : 2.1758
Epoch[ 15/ 100],Loss : 2.1750
Epoch[ 16/ 100],Loss : 2.1735
Epoch[ 17/ 100],Loss : 2.1720
Epoch[ 18/ 100],Loss : 2.1767
Epoch[ 19/ 100],Loss : 2.1717
Epoch[ 20/ 100],Loss : 2.1716
Epoch[ 21/ 100],Loss : 2.1762
Epoch[ 22/ 100],Loss : 2.1723
Epoch[ 23/ 100],Loss : 2.1716
Epoch[ 24/ 100],Loss : 2.1716
Epoch[ 25/ 100],Loss : 2.1762
Epoch[ 26/ 100],Loss : 2.1717
Epoch[ 27/ 100],Loss : 2.1716
Epoch[ 28/ 100],Loss : 2.1762
Epoch[ 29/ 100],Loss : 2.1716
Epoch[ 30/ 100],Loss : 2.1716
Epoch[ 31/ 100],Loss : 2.1743
Epoch[ 32/ 100],Loss : 2.1716
Epoch[ 33/ 100],Loss : 2.1716
Epoch[ 34/

In [29]:
# 일반화 평가 
test_dataset = TensorDataset(test_scaled,test_target)
test_loader = DataLoader(test_dataset,batch_size=batch_size)
# 평가하기
test_loss,test_accuracy = evaluate(model,test_loader,criterion,device)
print(f"Loss : {test_loss}, Accuracy : {test_accuracy}")

Loss : 2.1766876578330994, Accuracy : 0.964035964035964


In [31]:
# 예측 
predictions = predict(model,test_loader,device)

In [32]:
# 결과
print(test_target[:10])
print(predictions[:10])

tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9])
[np.int64(7), np.int64(2), np.int64(1), np.int64(0), np.int64(4), np.int64(1), np.int64(4), np.int64(9), np.int64(6), np.int64(9)]


---
#### 이미지를 불러와서 predict 해보기

In [34]:
from PIL import Image

In [35]:
# Image 불러오기
img = Image.open("../Data/mnist_test_3.jpg")
img

In [36]:
import numpy as np

In [37]:
# image -> numpy array
imgArray = np.array(img)
imgArray

array([[  0,   0,   0,   0,   0,   0,   0,   0,   3,   0,   0,  15,   6,
          0,   8,   0,   9,   0,   7,   0,   0,   0,   1,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,  11,   2,   0,   0,
          0,  18,   0,   0,   0,   0,   0,   0,   0,   0,   5,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  16,  19,
          0,   0,   5,   2,  18,   0,  18,   3,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,  13,   0,   0,   4,
          0,   0,   0,   6,   0,   0,   7,   0,   0,   7,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  11,   3,   1,  64, 129,
        143, 170, 204, 107,   0,   0,   1,   0,   0,   6,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  28, 164, 254, 255, 236,
        235, 255, 249, 237,  79,   2,   6,   8,   0,   0,   0,   0,   0,
          0,   0],
       [  

In [38]:
imgArray2 = imgArray.reshape(1, -1)
imgArray2

array([[  0,   0,   0,   0,   0,   0,   0,   0,   3,   0,   0,  15,   6,
          0,   8,   0,   9,   0,   7,   0,   0,   0,   1,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  11,   2,
          0,   0,   0,  18,   0,   0,   0,   0,   0,   0,   0,   0,   5,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,  16,  19,   0,   0,   5,   2,  18,   0,  18,   3,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,  13,   0,   0,   4,   0,   0,   0,   6,   0,   0,   7,
          0,   0,   7,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,  11,   3,   1,  64, 129, 143, 170, 204, 107,   0,
          0,   1,   0,   0,   6,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,  28, 164, 254, 255, 236, 235, 255, 249,
        237,  79,   2,   6,   8,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0, 191, 177

In [39]:
# numpy -> torch
img_input = torch.tensor(imgArray2)
img_input.shape

torch.Size([1, 784])

In [40]:
# 정규화
img_input = img_input / 255.
img_input.shape

torch.Size([1, 784])

In [41]:
# 단일 데이터의 예측 함수
def predictOne(model,inputs, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        predictions.extend(predicted.cpu().numpy())
    return predictions

In [42]:
predictOne(model, img_input, device)

[np.int64(3)]